# Build the public release (jev-clinical-calibration)

**No model calls; about 1 to 2 minutes.** Assembles `MyDrive/jev_bench/release/jev-clinical-calibration/` and a zip of it, ready to upload to GitHub:

- README, license and citation file (from `release_static/`), and the study changelog
- every harness version (`code/v3` to `code/v8`) plus the verified final set in `code/current/`
- every Colab notebook
- the DDXPlus, ophthalmic and uninformative-input case files; for Semigran, only case identifiers, diagnoses and triage levels (its text is not redistributed)
- all results compressed, with two safety checks: the build stops if anything resembling an API key appears, and any stored response that contains Semigran vignette text has that field removed
- the summary, figures, run manifest and reasoning-token audit

It ends by listing every file and flagging anything over GitHub's 25 MB browser-upload limit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, json, gzip, csv, glob, hashlib, re, subprocess, zipfile
WORK = '/content/drive/MyDrive/jev_bench'
STATIC = f'{WORK}/release_static'
REL = f'{WORK}/release/jev-clinical-calibration'
CURRENT = {'clients.py': ('v5', '16472190f65fd6ec55e2888b89c8daf9176768a90ac5c191e13ebfb8a79f49d0'), 'ddxplus_to_cases.py': ('v5', '0a2c9072ec480fe52e3eef4f420fcc2f37a71372d252c84a442e34a922912c51'), 'plots.py': ('v6', 'b1b426a07e94fbb615d116c39ff0604d7aa2a9b9d0b8e4b9d0718efeebc35d1c'), 'run.py': ('v7', 'b1a46cc7b071d9603600bb2940079090f09ad11d50062599a9578c3aec87b798'), 'analyze.py': ('v8', '74d9356d5865eb7ba7bc6c56e3b0ed0f036a072f1cefe2ba71b8e11a0ea60921')}
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()

if os.path.exists(REL):
    shutil.rmtree(REL)
for d in ('code/current', 'notebooks', 'data', 'results/figures'):
    os.makedirs(f'{REL}/{d}', exist_ok=True)

# 1. front matter
for f in ('README.md', 'LICENSE', 'CITATION.cff'):
    shutil.copy(f'{STATIC}/{f}', f'{REL}/{f}')
shutil.copy(f'{WORK}/CHANGELOG.md', f'{REL}/CHANGELOG.md')

# 2. code: every version folder, plus the verified final set
for vdir in sorted(glob.glob(f'{WORK}/code/v*')):
    shutil.copytree(vdir, f'{REL}/code/{os.path.basename(vdir)}')
for f, (v, h) in CURRENT.items():
    p = f'{WORK}/code/{v}/{f}'
    assert sha(p) == h, f'hash mismatch: {v}/{f}'
    shutil.copy(p, f'{REL}/code/current/{f}')

# 3. notebooks
for p in sorted(glob.glob(f'{WORK}/notebooks/*.ipynb')):
    shutil.copy(p, f'{REL}/notebooks/')

# 4. data (Semigran text is not redistributed)
for f in ('ddxplus_cases.csv', 'ddxplus_options.json', 'zandi_cases.csv', 'zandi_options.json',
          'vague.csv', 'options.json', 'SIGNOFFS.md', 'OSF_REGISTRATION.md'):
    shutil.copy(f'{WORK}/data/{f}', f'{REL}/data/{f}')
sg = list(csv.DictReader(open(f'{WORK}/data/cases.csv', newline='')))
keep = ['case_id', 'dataset', 'true_dx', 'triage_gold', 'source_ref', 'two_column_reconstructed']
with open(f'{REL}/data/semigran_case_index.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=keep); w.writeheader()
    for r in sg:
        w.writerow({k: r.get(k, '') for k in keep})

# 5. results, with two safety checks: no Semigran text, no credentials
snippets = [r['text'][:40] for r in sg] + [r['full_vignette'][:40] for r in sg if r.get('full_vignette')]
cred = re.compile(r'sk-[A-Za-z0-9_\-]{16,}|Bearer\s+[A-Za-z0-9_\-]{16,}')
n = stripped = 0
with open(f'{WORK}/results.jsonl') as src, gzip.open(f'{REL}/results/results.jsonl.gz', 'wt') as dst:
    for line in src:
        if cred.search(line):
            raise SystemExit('Credential-like string found in results; stopping. Tell Claude.')
        if any(s in line for s in snippets):
            r = json.loads(line); r.pop('raw', None); r['raw_removed'] = 'contained Semigran text'
            line = json.dumps(r) + '\n'; stripped += 1
        dst.write(line); n += 1
print(f'results: {n} rows written, {stripped} raw fields removed for Semigran text')
for f in ('summary.md', 'MANIFEST.json', 'opus55_thinking_audit.json'):
    if os.path.exists(f'{WORK}/runs/{f}'):
        shutil.copy(f'{WORK}/runs/{f}', f'{REL}/results/{f}')
for p in glob.glob(f'{WORK}/runs/figures/*'):
    shutil.copy(p, f'{REL}/results/figures/')

# 6. zip + report
zpath = f'{WORK}/release/jev-clinical-calibration.zip'
if os.path.exists(zpath):
    os.remove(zpath)
files = sorted(p for p in glob.glob(f'{REL}/**/*', recursive=True) if os.path.isfile(p))
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, os.path.relpath(p, os.path.dirname(REL)))
big = [(os.path.relpath(p, REL), os.path.getsize(p)) for p in files if os.path.getsize(p) > 25e6]
print(f'{len(files)} files, zip {os.path.getsize(zpath) / 1e6:.1f} MB')
print('files over 25 MB (GitHub web upload limit):', big or 'none')
for p in files:
    print(f'  {os.path.getsize(p) / 1e3:9.1f} KB  {os.path.relpath(p, REL)}')
